# MACURA learns a Backflip

Pogo (a one-legged pogo-stick gymnast) learns to backflip. A backflip is an
aggressive, unstable maneuver: a fixed-horizon rollout (MBPO) over-imagines the
flight and Pogo face-plants, while MACURA truncates the imagined rollout when the
model gets uncertain and learns the flip from trustworthy pieces. We train the four
algorithms (MACURA, MBPO, M2AC, SAC) - shared SAC backbone and ensemble, only the
rollout differs - then watch them fly. Run top to bottom on a GPU runtime.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/macura-backflip"
BRANCH = "main"
ROOT   = "/content/macura-backflip"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Setup**

In [ ]:
from src.bootstrap import setup
setup(drive=True, install_deps=True)

In [ ]:
# imports
%run /content/macura-backflip/src/imports.py
set_seed(cfg.SEED)

## **Meet Pogo**\nUntrained, it just flails - model error compounds fast here.

In [ ]:
env, obs_dim, act_dim = make_env(cfg.ENV, seed=0, render=True)
render_filmstrip(env, policy="random", title="Random Pogo");

## **Train**\nFour algorithms x seeds. A run whose log already exists on Drive is reloaded.

In [ ]:
runs = []
for algo in cfg.ALGORITHMS:
    for seed in cfg.SEEDS:
        log = f"{cfg.DRIVE_ROOT}/logs/{algo}_seed{seed}.json"
        if os.path.exists(log):
            runs.append(json.load(open(log)))
        else:
            runs.append(train_one(algo, cfg.CFG, cfg.DRIVE_ROOT, seed))

## **Results**\nSample efficiency, backflips landed (headline), faceplant rate, final quality, MACURA signature.

In [ ]:
plot_sample_efficiency(runs)
plot_success_rate(runs)
plot_failure_rate(runs)
plot_final_quality(runs)
plot_rollout_depth([r for r in runs if r["algo"] == "macura"][0]);

## **Flight demo**

In [ ]:
videos = record_best_videos(cfg.CFG, cfg.DRIVE_ROOT, device, seeds=cfg.SEEDS, seconds=8)
for name, path in videos.items():
    print(name)
    display(Video(path, embed=True, width=360))

In [ ]:
labels = {name.split("_seed")[0]: path for name, path in videos.items()}
if labels:
    display(Video(stitch_videos_grid(labels, f"{cfg.DRIVE_ROOT}/videos/reel.mp4", cols=2), embed=True, width=520))

## **Save**\nMeasured ordering. Watch them live on your Mac: `mjpython run_live_mac.py --compare runs/checkpoints`.

In [ ]:
for r in sorted(runs, key=lambda r: -r["final_eval"]["eval_return"]):
    fe = r["final_eval"]
    print(f'{r["algo"]:7s} seed{r["seed"]}  return {fe["eval_return"]:7.1f}'
          f'  land {fe["eval_success_rate"]:.2f}  fail {fe["eval_failure_rate"]:.2f}')